In [1]:
from langgraph.graph import StateGraph, START, END 
from langchain_openai import ChatOpenAI 
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
model = ChatOpenAI()

In [4]:
class BlogState(TypedDict):
    topic: str
    outline: str
    blog_post: str

In [5]:
def create_outline(state: BlogState) -> BlogState:
   
    # fetch title from state
    title = state['topic']
    
    #call llm gen outline
    prompts= f"generate a detailed outline for a blog post about {title}."
    outline= model.invoke(prompts).content
    
    # update state
    state['outline'] = outline
    
    return state
    

In [6]:
def create_blog(state: BlogState) -> BlogState:
    
    # fetch title and outline from state
    title = state['topic']
    outline = state['outline']
    
    #call llm gen blog post
    prompts= f"write a detailed blog post about {title} based on the following outline: {outline}."
    blog_post= model.invoke(prompts).content
    
    # update state
    state['blog_post'] = blog_post
    
    return state

In [10]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow= graph.compile()

In [ ]:
initial_state = {"topic": "The Future of Artificial Intelligence"}
final_state = workflow.invoke(initial_state)

print(final_state)

In [ ]:
print(final_state['outline'])

In [ ]:
print(final_state['blog_post'])